# 02 — Experiments: role database + CV skill extraction

Picks up the model trained in `01_training.ipynb`, runs it over job postings to build a **role -> top skills** database, and tries it on a sample CV. Ends by saving the final artifacts (model, tokenizer, tag list, role database) that `03_testing.ipynb` and the API consume.

Run `01_training.ipynb` first — this notebook expects a `skill_ner_distilbert_best/` checkpoint to already exist next to it.

In [2]:
import pandas as pd
import numpy as np
import re
import json
from collections import Counter

import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(42)
np.random.seed(42)


Device: cuda


## Reload the trained model

Tag list is reconstructed from the model config, since `id2label`/`label2id` were baked in when the model was saved in `01_training.ipynb`.

In [3]:
tokenizer = AutoTokenizer.from_pretrained("../Models/skill_ner_distilbert_best")
model = AutoModelForTokenClassification.from_pretrained("../Models/skill_ner_distilbert_best").to(device)
model.eval()

MAX_LEN = 100  # must match the value used during training
id2label = model.config.id2label
TAG_LIST = [id2label[i] for i in range(len(id2label))]
tag_to_idx = {t: i for i, t in enumerate(TAG_LIST)}
O_IDX = tag_to_idx["O"]
print("Tags:", TAG_LIST)

def tokenize(text):
    return re.findall(r"[a-z0-9+#.]+", str(text).lower())


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Tags: ['O', 'B-TECH', 'I-TECH', 'B-SOFT', 'I-SOFT', 'B-TOOL', 'I-TOOL', 'B-DOMAIN', 'I-DOMAIN', 'B-CERT', 'I-CERT']


## Stage 3 (recap) — `extract_skills()`

Same logic as training-time inference, wrapped for reuse.

In [4]:
def tags_to_spans(tags):
    spans = []
    start, cat = None, None
    for i, t in enumerate(list(tags) + [O_IDX]):
        label = TAG_LIST[t]
        if label.startswith("B-"):
            if start is not None:
                spans.append((start, i, cat))
            start, cat = i, label[2:]
        elif label.startswith("I-") and cat == label[2:]:
            continue
        else:
            if start is not None:
                spans.append((start, i, cat))
            start, cat = None, None
    return spans

def extract_skills(text):
    words = tokenize(text)
    if not words:
        return []

    enc = tokenizer(
        words, is_split_into_words=True, truncation=True,
        max_length=MAX_LEN, padding="max_length", return_tensors="pt",
    )
    word_ids = enc.word_ids(batch_index=0)

    model.eval()
    with torch.no_grad():
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        ).logits
    preds = logits.argmax(-1)[0].cpu().tolist()

    word_tags = [O_IDX] * len(words)
    seen = set()
    for pos, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        word_tags[wid] = preds[pos]

    spans = tags_to_spans(word_tags)
    return [{"skill": " ".join(words[s:e]), "type": cat} for s, e, cat in spans]


extract_skills("experienced python developer with docker, kubernetes and aws")


[{'skill': 'python', 'type': 'TECH'},
 {'skill': 'docker', 'type': 'TECH'},
 {'skill': 'kubernetes', 'type': 'TECH'},
 {'skill': 'aws', 'type': 'TECH'}]

## Reload postings and reproduce the train/val split

Same CSV, same filter, same `train_test_split(..., random_state=42)` as `01_training.ipynb`, so `idx_train`/`idx_val` line up with the postings the model was actually trained/validated on (i.e. not the held-out test set).

In [5]:
df = pd.read_csv('../data/processed/data_tech_only.csv')
df = df[df["skill_count"] > 0].dropna(subset=["cleaned_text"]).reset_index(drop=True)

idx_train, idx_temp = train_test_split(df.index, test_size=0.3, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42)
print(f"train: {len(idx_train)}  val: {len(idx_val)}  test: {len(idx_test)}")


train: 4795  val: 1027  test: 1028


## Stage 3 — Build the role database

In [6]:
def build_role_database(dataframe, title_col="cleaned_title", top_k=15, min_postings=1):
    role_db = {}
    for role, group in dataframe.groupby(title_col):
        if len(group) < min_postings:
            continue
        counter = Counter()
        for text in group["cleaned_text"]:
            for item in extract_skills(text):
                counter[(item["skill"], item["type"])] += 1
        top = counter.most_common(top_k)
        role_db[role] = [{"skill": s, "type": t, "freq": f} for (s, t), f in top]
    return role_db


# Build the role database from postings the model was trained/validated on
role_db = build_role_database(df.loc[idx_train.union(idx_val)])
print(f"Roles in database: {len(role_db)}")
sample_role = max(role_db, key=lambda r: sum(s['freq'] for s in role_db[r]))
print(f"\nExample — '{sample_role}':")
for s in role_db[sample_role]:
    print(f"  {s['skill']:<25} ({s['type']})  seen in {s['freq']} posting(s)")


Roles in database: 2545

Example — 'android developer':
  android                   (TECH)  seen in 284 posting(s)
  kotlin                    (TECH)  seen in 125 posting(s)
  java                      (TECH)  seen in 111 posting(s)
  android sdk               (TECH)  seen in 48 posting(s)
  oop                       (TECH)  seen in 20 posting(s)
  git                       (TOOL)  seen in 17 posting(s)
  rxjava                    (TECH)  seen in 14 posting(s)
  retrofit                  (TECH)  seen in 14 posting(s)
  coroutines                (TECH)  seen in 14 posting(s)
  dagger                    (TECH)  seen in 14 posting(s)
  mvvm                      (TECH)  seen in 12 posting(s)
  fintech                   (DOMAIN)  seen in 9 posting(s)
  ios                       (TECH)  seen in 9 posting(s)
  design patterns           (TECH)  seen in 9 posting(s)
  android jetpack           (TECH)  seen in 8 posting(s)


## Stage 4 — CV -> Extract skills

In [7]:
cv_text = """
Experienced backend developer with 4 years building REST APIs in Python and Node.js.
Strong SQL and PostgreSQL skills, some exposure to Docker and Git-based workflows.
"""

cv_skills = extract_skills(cv_text)
cv_skills


[{'skill': 'python', 'type': 'TECH'},
 {'skill': 'sql', 'type': 'TECH'},
 {'skill': 'postgresql', 'type': 'TECH'},
 {'skill': 'git', 'type': 'TOOL'}]

## Save artifacts

Model + tokenizer, tag list, and the role database — everything needed to run the pipeline (extract -> compare -> recommend) again without retraining. `03_testing.ipynb` and the FastAPI app load from these paths.

In [8]:
import os
os.makedirs("../Models", exist_ok=True)

model.save_pretrained("../Models/skill_ner_distilbert")
tokenizer.save_pretrained("../Models/skill_ner_distilbert")

with open("../Models/tags.json", "w") as f:
    json.dump(TAG_LIST, f)

with open("../Models/role_database.json", "w") as f:
    json.dump(role_db, f, indent=2)

print("Saved: ../Models/skill_ner_distilbert/ (model + tokenizer), tags.json, role_database.json")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: ../Models/skill_ner_distilbert/ (model + tokenizer), tags.json, role_database.json
